In [ ]:
# YouTube Language Query — RAG over video transcripts

Run all cells in order. Copy `.env.example` to `.env` and add your OpenRouter API key before using the LLM cells.

In [ ]:
!pip install -q \
    python-dotenv \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-openai \
    faiss-cpu \
    sentence-transformers

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # OPENAI_API_KEY, OPENAI_BASE_URL from .env

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

video_id = "CAgWNxlmYsc"

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_obj = ytt_api.fetch(video_id)
    full_text = " ".join(chunk.text for chunk in transcript_obj)
    print(full_text)

except NoTranscriptFound:
    transcript_list = ytt_api.list(video_id)
    transcript = next(iter(transcript_list))
    transcript_obj = transcript.fetch()
    full_text = " ".join(chunk.text for chunk in transcript_obj)
    print(full_text)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [ ]:
print(transcript_obj)

In [ ]:
!pip install transformers sentencepiece

In [ ]:
!pip install langdetect langcodes language_data transformers sentencepiece

In [ ]:
from langdetect import detect
import langcodes
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

def detect_and_translate(text):
    lang_code = detect(text)
    lang_name = langcodes.get(lang_code).display_name()
    print(f"Detected Language: {lang_name} ({lang_code})")

    if lang_code == "en":
        print("Already in English. No translation needed.")
        return text

    model_name = f"Helsinki-NLP/opus-mt-{lang_code}-en"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)

    chunks = [text[i:i+400] for i in range(0, len(text), 400)]
    translated_chunks = []
    for chunk in tqdm(chunks, desc="Translating"):
        tokens = tokenizer([chunk], return_tensors="pt", padding=True, truncation=True)
        translated = model.generate(**tokens)
        translated_chunks.append(tokenizer.decode(translated[0], skip_special_tokens=True))

    translated = " ".join(translated_chunks)
    print(f"Translated: {translated}")
    return translated

a = detect_and_translate(full_text)
print(a)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.create_documents([full_text])

In [ ]:
len(chunks)

In [ ]:
chunks[60]

In [ ]:
!pip install sentence-transformers

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Free local embeddings (runs fully offline)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector database
vector_store = FAISS.from_documents(chunks, embeddings)

print("FAISS index created successfully 🚀")

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind')

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-3.5-turbo",  # this usually always exists
    temperature=0.2
)

print(llm.invoke("Say hello").content)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)